# ConfRover overfit-one-batch smoke test

Goal: verify that the training plumbing in `confrover.train` is wired correctly by overfitting a single trajectory window to a low loss. If loss drops monotonically by ~2 orders of magnitude over a few hundred steps, the training loop is healthy.

Prereqs:
- An interactive Phoenix GPU session (see `scripts/phoenix_interactive.sh`).
- The conda env from the README installed.
- OpenFold features cached for the test protein (the bundled `tests/test_data/openfold_repr/` already covers `7jfl_C`).

Run cells top-to-bottom. The whole thing should take 10–20 minutes on an H100.

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Repo root:', REPO_ROOT)

CACHE = Path(os.environ.get('CONFROVER_CACHE_DIR', Path.home() / 'scratch' / 'confrover_cache')).expanduser()
CACHE.mkdir(parents=True, exist_ok=True)
print('Cache dir:', CACHE)

import torch
assert torch.cuda.is_available(), 'No GPU visible \u2014 check salloc / CUDA_VISIBLE_DEVICES'
torch.set_float32_matmul_precision('high')
print('GPU:', torch.cuda.get_device_name(0))

## 1. Build the dataset

We use the bundled ATLAS test protein (`7jfl_C`) and its pre-computed OpenFold repr.

In [ ]:
from confrover.train.dataset import TrajDataset
from confrover.data.pretrain_repr.openfold.loader import OpenFoldReprLoader

repr_loader = OpenFoldReprLoader(
    repr_root=str(REPO_ROOT / 'tests' / 'test_data' / 'openfold_repr'),
    num_recycles=3,
    load_single=True,
    load_pair=True,
    v1=False,
)

dataset = TrajDataset(
    config=str(REPO_ROOT / 'examples' / 'train_manifest_smoke.json'),
    repr_loader=repr_loader,
    relpath_to=str(REPO_ROOT),
    deterministic=True,         # always return the same window
    batch_size=1,
    num_workers=0,
    shuffle=False,
)
print('Cases:', [c.case_id for c in dataset.cfg.cases])

item = dataset[0]
for k, v in item.items():
    if hasattr(v, 'shape'):
        print(f'  {k:30s} shape={tuple(v.shape)} dtype={v.dtype}')
    else:
        print(f'  {k:30s} = {v}')

In [ ]:
batch = TrajDataset.collate([dataset[0]])
print('Collated batch:')
for k, v in batch.items():
    if hasattr(v, 'shape'):
        print(f'  {k:30s} shape={tuple(v.shape)}')
    else:
        print(f'  {k:30s} = {v}')

## 2. Build the model from the training config

In [ ]:
from omegaconf import OmegaConf
import hydra

model_cfg = OmegaConf.load(REPO_ROOT / 'src' / 'confrover' / 'configs' / 'model' / 'confrover_train.yaml')
model_cfg.optimizer_cfg.lr = 5e-4    # high LR for an aggressive overfit
model_cfg.scheduler_cfg.warmup_steps = 0
model_cfg.scheduler_cfg.total_steps = 0

model = hydra.utils.instantiate(model_cfg)
model = model.to('cuda')

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params/1e6:.2f}M')

## 3. Move the batch to GPU once and run a single forward pass

If anything is going to crash, it crashes here. Step through the error trace before going further.

In [ ]:
from lightning.pytorch.utilities import move_data_to_device

batch_gpu = move_data_to_device(batch, 'cuda')
loss, aux = model._shared_step(batch_gpu, stage='train')
print('First-pass loss:', loss.item())
print('Aux info:')
for k, v in aux.items():
    print(f'  {k:20s} = {float(v):.4f}')

## 4. Manual overfit loop

We deliberately bypass the Lightning Trainer here — a hand-rolled loop is the easiest way to plot loss as it drops.

In [ ]:
import math

model.train()
opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=model.optimizer_cfg['lr'],
    weight_decay=0.0,
)

N_STEPS = 500
loss_history = []

for step in range(N_STEPS):
    opt.zero_grad(set_to_none=True)
    loss, aux = model._shared_step(batch_gpu, stage='train')
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    opt.step()
    loss_history.append(float(loss.item()))
    if step % 25 == 0 or step == N_STEPS - 1:
        print(f'step {step:4d}  loss={loss_history[-1]:.4f}  '
              f'rot={float(aux["loss_rot"]):.4f}  trans={float(aux["loss_trans"]):.4f}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(loss_history); ax[0].set_title('train/loss (linear)'); ax[0].set_xlabel('step')
ax[1].plot(loss_history); ax[1].set_yscale('log'); ax[1].set_title('train/loss (log)'); ax[1].set_xlabel('step')
plt.tight_layout(); plt.show()

print(f'\nstart loss   = {loss_history[0]:.4f}')
print(f'final loss   = {loss_history[-1]:.4f}')
print(f'ratio        = {loss_history[0] / max(loss_history[-1], 1e-8):.2f}x decrease')

print('\nSmoke test passes if:')
print('  * Loss decreases monotonically (modulo small noise from t sampling).')
print('  * Final loss is at least 5–10x lower than starting loss.')
print('  * No NaNs in the loss trace.')

## 5. (Optional) Sanity-check: sample from the overfit model

After overfitting, generated samples should look *much* like the training frame. This is qualitative — quantitative metrics come later.

In [ ]:
# Switch to eval mode and run inference on the same protein.
# (We reuse the upstream generate() API, which expects a sampler; the training
# config doesn't instantiate one, so we attach an EulerSampler here.)
from confrover.model.decoder.confdiff.sampler.euler import EulerSampler
model.decoder.sampler = EulerSampler(diffusion_steps=20)
model.eval()

out = model.generate(
    case_id='7jfl_C',
    seqres='SALQDLLRTLKSPSSPQQQQQVLNILKSNPQLMAAFIKQRTAKYVAN',
    task_mode='iid',
    output_dir=str(CACHE / 'smoke_iid_overfit'),
    n_replicates=2,
    cache_dir=str(CACHE),
)
print('Output dir:', out)